# Enriched MILP Ideas Ablation

Тест гипотез на уровне solver-ов (без внешних fallback/repair).

In [1]:
from __future__ import annotations

from pathlib import Path
from datetime import datetime
import json
import sys
import pandas as pd

REPO_ROOT = Path.cwd().resolve().parent if Path.cwd().name == 'demo' else Path.cwd().resolve()
SRC_DIR = REPO_ROOT / 'src'
if str(SRC_DIR) not in sys.path:
    sys.path.insert(0, str(SRC_DIR))

from flowopt.solvers.enriched import (
    solve_enriched_milp_ablation_baseline,
    solve_enriched_milp_ablation_adaptive_k,
    solve_enriched_milp_ablation_penalty_sweep,
    solve_enriched_milp_ablation_zone_bundle,
    solve_enriched_milp_ablation_portfolio,
    solve_enriched_batched_greedy,
)


In [2]:
DATASET_PATH = REPO_ROOT / 'demo' / 'data' / 'object_mass_feasible_fullfleet' / 'container_full_split_all_agents' / 'sweeps_task_agent_5pct' / 'dataset_real_spb_clean_full_split_by_containers_all_agents_with_distances_t001_a100.json'

if not DATASET_PATH.exists():
    raise FileNotFoundError(f'Not found: {DATASET_PATH}')

print('REPO_ROOT   :', REPO_ROOT)
print('DATASET_PATH:', DATASET_PATH)


REPO_ROOT   : /Users/igoreshka/Desktop/Optimization-of-flows
DATASET_PATH: /Users/igoreshka/Desktop/Optimization-of-flows/demo/data/object_mass_feasible_fullfleet/container_full_split_all_agents/sweeps_task_agent_5pct/dataset_real_spb_clean_full_split_by_containers_all_agents_with_distances_t001_a100.json


In [3]:
EXPERIMENTS = [
    {
        'exp_name': 'idea_baseline_strict_t30_k80',
        'fn': solve_enriched_milp_ablation_baseline,
        'kwargs': dict(time_limit_sec=30, max_pairs_per_task=80, unassigned_penalty=1e6),
    },
    {
        'exp_name': 'idea_adaptive_k_t90',
        'fn': solve_enriched_milp_ablation_adaptive_k,
        'kwargs': dict(time_budget_sec=90.0, pair_schedule=(40, 80, 120, 200), unassigned_penalty=1e6),
    },
    {
        'exp_name': 'idea_penalty_sweep_t90',
        'fn': solve_enriched_milp_ablation_penalty_sweep,
        'kwargs': dict(time_budget_sec=90.0, max_pairs_per_task=160, penalty_schedule=(1e5, 1e6, 1e7, 1e8)),
    },
    {
        'exp_name': 'idea_zone_bundle_tasks',
        'fn': solve_enriched_milp_ablation_zone_bundle,
        'kwargs': dict(time_limit_sec_per_zone=20, max_pairs_per_bundle=120, bundle_fill_factor=0.90, bundle_max_tasks=8, objective='tasks'),
    },
    {
        'exp_name': 'idea_portfolio_t60',
        'fn': solve_enriched_milp_ablation_portfolio,
        'kwargs': dict(time_budget_sec=60.0, max_starts=12, per_start_time_limit_sec=8, max_pairs_per_task=80, jitter=0.08, seed=42),
    },
    {
        'exp_name': 'reference_batched_greedy',
        'fn': solve_enriched_batched_greedy,
        'kwargs': dict(top_k_agents=30, balance_penalty=0.02, random_seed=42),
    },
]


In [4]:
results = []

for cfg in EXPERIMENTS:
    exp_name = cfg['exp_name']
    fn = cfg['fn']
    kwargs = cfg['kwargs']
    print(f"[RUN] {exp_name}")
    res = fn(dataset_path=DATASET_PATH, **kwargs)
    d = res.as_dict()
    checks = (d.get('details') or {}).get('checks') or {}

    row = {
        'exp_name': exp_name,
        'algorithm': d.get('algorithm'),
        'feasible': d.get('feasible'),
        'assigned_tasks': d.get('assigned_routes'),
        'assigned_trips': d.get('assigned_trips'),
        'unassigned_tasks': d.get('unassigned_tasks'),
        'active_agents': d.get('active_agents'),
        'transport_work_ton_km': d.get('transport_work_ton_km'),
        'total_km': d.get('total_km'),
        'deadhead_share_pct': d.get('deadhead_share_pct'),
        'total_hours': d.get('total_hours'),
        'runtime_sec': d.get('runtime_sec'),
        'all_checks_ok': checks.get('all_checks_ok'),
        'daily_limits_ok': checks.get('daily_limits_ok'),
        'object_limits_ok': checks.get('object_limits_ok'),
        'compatibility_ok': checks.get('compatibility_ok'),
        'reachability_ok': checks.get('reachability_ok'),
        'overflow_km_agents': checks.get('overflow_km_agents'),
        'overflow_hours_agents': checks.get('overflow_hours_agents'),
        'object_mass_violations': checks.get('object_mass_violations'),
        'object_volume_violations': checks.get('object_volume_violations'),
        'solver_error': d.get('solver_error'),
    }
    results.append(row)
    print(f"[DONE] {exp_name}: feasible={row['feasible']} unassigned={row['unassigned_tasks']} runtime={row['runtime_sec']}s")

summary = pd.DataFrame(results)
summary = summary.sort_values(
    by=['all_checks_ok', 'unassigned_tasks', 'runtime_sec', 'total_km'],
    ascending=[False, True, True, True],
).reset_index(drop=True)
summary


[RUN] idea_baseline_strict_t30_k80


[DONE] idea_baseline_strict_t30_k80: feasible=False unassigned=188 runtime=30.92s
[RUN] idea_adaptive_k_t90


[DONE] idea_adaptive_k_t90: feasible=False unassigned=96 runtime=91.265s
[RUN] idea_penalty_sweep_t90


[DONE] idea_penalty_sweep_t90: feasible=False unassigned=96 runtime=91.133s
[RUN] idea_zone_bundle_tasks


[DONE] idea_zone_bundle_tasks: feasible=False unassigned=56 runtime=27.548s
[RUN] idea_portfolio_t60


[DONE] idea_portfolio_t60: feasible=False unassigned=90 runtime=63.671s
[RUN] reference_batched_greedy


[DONE] reference_batched_greedy: feasible=True unassigned=0 runtime=0.889s


,exp_name,algorithm,feasible,assigned_tasks,assigned_trips,unassigned_tasks,active_agents,transport_work_ton_km,total_km,deadhead_share_pct,...,all_checks_ok,daily_limits_ok,object_limits_ok,compatibility_ok,reachability_ok,overflow_km_agents,overflow_hours_agents,object_mass_violations,object_volume_violations,solver_error
0,reference_batched_greedy,enriched_batched_greedy_v1,True,1240,913,0,525,2392.824,54829.789,72.282,...,True,True,True,True,True,0,0,0,0,None
1,idea_zone_bundle_tasks,idea_milp_zone_bundle_v1,False,1184,882,56,340,2276.496,51251.494,72.038,...,False,True,True,True,True,0,0,0,0,None
2,idea_portfolio_t60,idea_milp_portfolio_v1,False,1150,1150,90,483,2128.128,70462.303,72.501,...,False,True,True,True,True,0,0,0,0,None
3,idea_penalty_sweep_t90,idea_milp_penalty_sweep_v1,False,1144,1144,96,409,2011.208,70519.626,72.610,...,False,True,True,True,True,0,0,0,0,None
4,idea_adaptive_k_t90,idea_milp_adaptive_k_v1,False,1144,1144,96,440,1929.577,70272.635,72.615,...,False,True,True,True,True,0,0,0,0,None
5,idea_baseline_strict_t30_k80,idea_milp_baseline_strict_v1,False,1052,1052,188,295,2037.226,57318.301,72.036,...,False,True,True,True,True,0,0,0,0,None


In [5]:
TIME_ABLATION = []

for budget in [20, 40, 80]:
    for name, fn, kwargs in [
        ('baseline', solve_enriched_milp_ablation_baseline, dict(time_limit_sec=budget, max_pairs_per_task=120)),
        ('adaptive_k', solve_enriched_milp_ablation_adaptive_k, dict(time_budget_sec=float(budget), pair_schedule=(40, 80, 120, 200))),
        ('portfolio', solve_enriched_milp_ablation_portfolio, dict(time_budget_sec=float(budget), max_starts=8, per_start_time_limit_sec=max(5, budget // 8), max_pairs_per_task=100, seed=42)),
    ]:
        res = fn(dataset_path=DATASET_PATH, **kwargs)
        d = res.as_dict()
        checks = (d.get('details') or {}).get('checks') or {}
        TIME_ABLATION.append({
            'method': name,
            'time_budget_sec': budget,
            'algorithm': d.get('algorithm'),
            'feasible': d.get('feasible'),
            'unassigned_tasks': d.get('unassigned_tasks'),
            'assigned_tasks': d.get('assigned_routes'),
            'runtime_sec': d.get('runtime_sec'),
            'total_km': d.get('total_km'),
            'all_checks_ok': checks.get('all_checks_ok'),
        })

abl_df = pd.DataFrame(TIME_ABLATION)
abl_df.sort_values(['method', 'time_budget_sec'])


,method,time_budget_sec,algorithm,feasible,unassigned_tasks,assigned_tasks,runtime_sec,total_km,all_checks_ok
1,adaptive_k,20,idea_milp_adaptive_k_v1,False,97,1143,23.988,69857.752,False
4,adaptive_k,40,idea_milp_adaptive_k_v1,False,97,1143,41.037,69857.752,False
7,adaptive_k,80,idea_milp_adaptive_k_v1,False,96,1144,81.021,70272.635,False
0,baseline,20,idea_milp_baseline_strict_v1,False,124,1116,21.025,66248.855,False
3,baseline,40,idea_milp_baseline_strict_v1,False,76,1164,41.152,70416.046,False
6,baseline,80,idea_milp_baseline_strict_v1,False,72,1168,80.937,70699.650,False
2,portfolio,20,idea_milp_portfolio_v1,False,93,1147,24.235,70369.264,False
5,portfolio,40,idea_milp_portfolio_v1,False,93,1147,41.639,70369.264,False
8,portfolio,80,idea_milp_portfolio_v1,False,89,1151,88.151,71059.028,False


In [6]:
pivot_cov = abl_df.pivot_table(index='time_budget_sec', columns='method', values='assigned_tasks', aggfunc='max')
pivot_unassigned = abl_df.pivot_table(index='time_budget_sec', columns='method', values='unassigned_tasks', aggfunc='min')

print('Assigned tasks by time budget:')
display(pivot_cov)
print('Unassigned tasks by time budget:')
display(pivot_unassigned)


Assigned tasks by time budget:


method,adaptive_k,baseline,portfolio
time_budget_sec,,,
20,1143,1116,1147
40,1143,1164,1147
80,1144,1168,1151


Unassigned tasks by time budget:


method,adaptive_k,baseline,portfolio
time_budget_sec,,,
20,97,124,93
40,97,76,93
80,96,72,89


In [7]:
OUT_DIR = REPO_ROOT / 'demo' / 'local' / 'enriched_milp_ideas_ablation'
OUT_DIR.mkdir(parents=True, exist_ok=True)
ts = datetime.now().strftime('%Y%m%d_%H%M%S')
out_path = OUT_DIR / f'ablation_{ts}.json'

payload = {
    'dataset_path': str(DATASET_PATH),
    'created_at': ts,
    'summary': results,
    'time_ablation': TIME_ABLATION,
}
out_path.write_text(json.dumps(payload, ensure_ascii=False, indent=2), encoding='utf-8')
print('Saved:', out_path)


Saved: /Users/igoreshka/Desktop/Optimization-of-flows/demo/local/enriched_milp_ideas_ablation/ablation_20260504_211458.json
